# GPT Decoder 与 KV Cache 从零实现：客服回复解码

## 面试问题

面试时我会先说明 KV cache 缓存的是每层历史 token 已投影后的 Key 和 Value，而当前步仍需计算 Query、Key、Value。prefill 一次处理完整提示词，decode 后续每步只输入新 token，再把新 K/V 追加到缓存。位置编号必须从缓存长度继续，因果 mask 也要允许新 Query 看见全部历史但不能看未来。验证实现最可靠的方法，是让全量重算与 cache 解码在同一权重、同一 token 上逐步比较 logits。cache 不改变模型质量，只减少重复 token 计算并增加显存占用。下面手写单层多头 Decoder、训练六条客服模式，并展示候选 logits、注意力、计算量和位置偏移 bug。

## 真实案例

语料是六条离线客服回复模板，每条七个可读 token，覆盖退款、物流、密码、发票、库存和优惠六种意图。模型只用于验证 decoder 与 cache 机制；语料极小、词表封闭，不代表语言能力，也不连接真实客服系统。

本实验是为了看清机制而构造的离线小样本，不代表线上收益，也不能外推到开放分布。

In [1]:
import math  # 导入平方根用于缩放点积注意力。
import torch  # 导入 PyTorch 以实现 Decoder 和张量缓存。
from torch import nn  # 导入神经网络基础层。
import torch.nn.functional as F  # 导入 softmax、交叉熵和激活函数。
torch.manual_seed(31)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小实验运行时间。
token_sequences = [["用户", "问", "退款", "助手", "答", "提交", "订单"], ["用户", "问", "物流", "助手", "答", "查询", "单号"], ["用户", "问", "密码", "助手", "答", "重置", "账号"], ["用户", "问", "发票", "助手", "答", "填写", "抬头"], ["用户", "问", "库存", "助手", "答", "查看", "商品"], ["用户", "问", "优惠", "助手", "答", "核对", "活动"]]  # 定义六条可读客服训练序列。
vocabulary = ["<pad>"] + sorted({token for sequence in token_sequences for token in sequence})  # 从离线语料构造封闭词表。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到整数编号的映射。
id_to_token = {index: token for token, index in token_to_id.items()}  # 建立整数编号到 token 的反向映射。
batch = torch.tensor([[token_to_id[token] for token in sequence] for sequence in token_sequences], dtype=torch.long)  # 把六条等长语料编码成 token 张量。
print("序号  完整客服序列")  # 打印真实语料预览表头。
for index, sequence in enumerate(token_sequences, start=1):  # 逐条展示训练语料而不是只显示整数张量。
    print(f"{index:02d}    {' / '.join(sequence)}")  # 输出当前客服序列的可读 token。
print(f"批次形状={tuple(batch.shape)}，词表大小={len(vocabulary)}，词表={vocabulary}")  # 汇总语料和词表规模。

序号  完整客服序列
01    用户 / 问 / 退款 / 助手 / 答 / 提交 / 订单
02    用户 / 问 / 物流 / 助手 / 答 / 查询 / 单号
03    用户 / 问 / 密码 / 助手 / 答 / 重置 / 账号
04    用户 / 问 / 发票 / 助手 / 答 / 填写 / 抬头
05    用户 / 问 / 库存 / 助手 / 答 / 查看 / 商品
06    用户 / 问 / 优惠 / 助手 / 答 / 核对 / 活动
批次形状=(6, 7)，词表大小=23，词表=['<pad>', '优惠', '助手', '单号', '发票', '商品', '填写', '密码', '库存', '抬头', '提交', '查看', '查询', '核对', '活动', '物流', '用户', '答', '订单', '账号', '退款', '重置', '问']


## 基线：每一步全量重算前缀

标准自回归解码每生成一个 token，就把不断增长的完整前缀再次送入模型。它是正确性基线：cache 版本必须产生相同 token 和近似相同 logits，区别只能是重复计算量。

## 手写核心：多头因果注意力、Decoder Block 与缓存拼接

下面只使用 Linear、Embedding、LayerNorm 等基础层，不导入 `TransformerDecoder` 或现成 GPT。注意力权重、K/V 张量和 logits 都会直接返回供观察。

In [2]:
class CausalSelfAttention(nn.Module):  # 定义支持历史 K/V 的手写多头因果注意力。
    def __init__(self, model_dim, head_count):  # 根据隐藏维度和头数创建投影矩阵。
        super().__init__()  # 初始化父类以注册所有参数。
        self.head_count = head_count  # 保存注意力头数供张量重排使用。
        self.head_dim = model_dim // head_count  # 计算每个注意力头的维度。
        self.query_projection = nn.Linear(model_dim, model_dim, bias=False)  # 创建 Query 线性投影。
        self.key_projection = nn.Linear(model_dim, model_dim, bias=False)  # 创建 Key 线性投影。
        self.value_projection = nn.Linear(model_dim, model_dim, bias=False)  # 创建 Value 线性投影。
        self.output_projection = nn.Linear(model_dim, model_dim, bias=False)  # 创建多头拼接后的输出投影。
    def forward(self, hidden, past_key=None, past_value=None):  # 计算当前 token 注意力并可追加历史缓存。
        batch_size, step_count, model_dim = hidden.shape  # 读取当前输入的批次、步数和隐藏维度。
        query = self.query_projection(hidden).view(batch_size, step_count, self.head_count, self.head_dim).transpose(1, 2)  # 投影并重排当前 Query。
        new_key = self.key_projection(hidden).view(batch_size, step_count, self.head_count, self.head_dim).transpose(1, 2)  # 投影并重排当前新增 Key。
        new_value = self.value_projection(hidden).view(batch_size, step_count, self.head_count, self.head_dim).transpose(1, 2)  # 投影并重排当前新增 Value。
        past_length = 0 if past_key is None else past_key.shape[2]  # 从历史缓存读取已经解码的 token 数。
        key = new_key if past_key is None else torch.cat([past_key, new_key], dim=2)  # 沿时间轴拼接历史和新增 Key。
        value = new_value if past_value is None else torch.cat([past_value, new_value], dim=2)  # 沿时间轴拼接历史和新增 Value。
        scores = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算缩放点积注意力分数。
        query_positions = torch.arange(past_length, past_length + step_count, device=hidden.device).view(1, 1, step_count, 1)  # 创建当前 Query 的绝对位置。
        key_positions = torch.arange(key.shape[2], device=hidden.device).view(1, 1, 1, key.shape[2])  # 创建缓存中全部 Key 的绝对位置。
        causal_mask = key_positions <= query_positions  # 只允许每个 Query 读取自己及更早位置。
        masked_scores = scores.masked_fill(~causal_mask, float("-inf"))  # 把未来位置分数设为负无穷。
        attention = torch.softmax(masked_scores, dim=-1)  # 在可见历史位置上归一化注意力。
        context = attention @ value  # 用注意力权重加权聚合 Value。
        merged = context.transpose(1, 2).contiguous().view(batch_size, step_count, model_dim)  # 把多头结果拼回隐藏维度。
        output = self.output_projection(merged)  # 混合多个注意力头的上下文。
        return output, (key, value), attention  # 返回上下文、新缓存和可解释权重。
class DecoderBlock(nn.Module):  # 定义一个预归一化 GPT Decoder Block。
    def __init__(self, model_dim, head_count):  # 创建注意力、归一化和前馈子层。
        super().__init__()  # 初始化父类以注册子模块。
        self.attention_norm = nn.LayerNorm(model_dim)  # 创建注意力前的 LayerNorm。
        self.attention = CausalSelfAttention(model_dim, head_count)  # 创建手写因果自注意力层。
        self.feedforward_norm = nn.LayerNorm(model_dim)  # 创建前馈网络前的 LayerNorm。
        self.feedforward_up = nn.Linear(model_dim, model_dim * 2)  # 把隐藏维度扩展两倍。
        self.feedforward_down = nn.Linear(model_dim * 2, model_dim)  # 把激活结果投影回模型维度。
    def forward(self, hidden, cache=None):  # 计算注意力与前馈残差并更新缓存。
        past_key = None if cache is None else cache[0]  # 从可选缓存中读取历史 Key。
        past_value = None if cache is None else cache[1]  # 从可选缓存中读取历史 Value。
        attention_output, new_cache, attention = self.attention(self.attention_norm(hidden), past_key, past_value)  # 运行支持缓存的因果注意力。
        hidden = hidden + attention_output  # 加入注意力残差连接。
        feedforward = self.feedforward_down(F.gelu(self.feedforward_up(self.feedforward_norm(hidden))))  # 计算两层 GELU 前馈网络。
        hidden = hidden + feedforward  # 加入前馈残差连接。
        return hidden, new_cache, attention  # 返回更新后的隐藏状态、缓存和注意力。
class TinyGPT(nn.Module):  # 定义单层教学版 GPT 语言模型。
    def __init__(self, vocabulary_size, model_dim=32, head_count=4, max_length=16):  # 创建 embedding、Decoder 和语言模型头。
        super().__init__()  # 初始化父类以注册全部参数。
        self.token_embedding = nn.Embedding(vocabulary_size, model_dim)  # 创建 token embedding 表。
        self.position_embedding = nn.Embedding(max_length, model_dim)  # 创建可学习绝对位置 embedding。
        self.block = DecoderBlock(model_dim, head_count)  # 创建手写 GPT Decoder Block。
        self.final_norm = nn.LayerNorm(model_dim)  # 创建输出 logits 前的 LayerNorm。
        self.language_head = nn.Linear(model_dim, vocabulary_size, bias=False)  # 把隐藏状态投影到词表 logits。
    def forward(self, input_ids, cache=None, position_offset=None):  # 支持全量训练和增量缓存两种前向方式。
        past_length = 0 if cache is None else cache[0].shape[2]  # 从缓存长度推导下一个绝对位置。
        start_position = past_length if position_offset is None else position_offset  # 默认续接位置并允许复现错误偏移。
        positions = torch.arange(start_position, start_position + input_ids.shape[1], device=input_ids.device)  # 创建当前输入 token 的绝对位置编号。
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions).unsqueeze(0)  # 合并 token 与位置 embedding。
        hidden, new_cache, attention = self.block(hidden, cache)  # 通过手写 Decoder Block 并更新 K/V。
        logits = self.language_head(self.final_norm(hidden))  # 输出每个当前位置的词表 logits。
        return logits, new_cache, attention  # 返回 logits、逐层缓存和注意力权重。
gpt = TinyGPT(len(vocabulary))  # 实例化封闭词表上的教学版 GPT。
print(gpt)  # 展示完整手写 Decoder 结构。
print(f"可训练参数量={sum(parameter.numel() for parameter in gpt.parameters())}")  # 输出教学模型参数规模。

TinyGPT(
  (token_embedding): Embedding(23, 32)
  (position_embedding): Embedding(16, 32)
  (block): DecoderBlock(
    (attention_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (attention): CausalSelfAttention(
      (query_projection): Linear(in_features=32, out_features=32, bias=False)
      (key_projection): Linear(in_features=32, out_features=32, bias=False)
      (value_projection): Linear(in_features=32, out_features=32, bias=False)
      (output_projection): Linear(in_features=32, out_features=32, bias=False)
    )
    (feedforward_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (feedforward_up): Linear(in_features=32, out_features=64, bias=True)
    (feedforward_down): Linear(in_features=64, out_features=32, bias=True)
  )
  (final_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (language_head): Linear(in_features=32, out_features=23, bias=False)
)
可训练参数量=10464


In [3]:
optimizer = torch.optim.Adam(gpt.parameters(), lr=0.03)  # 创建优化器学习六条客服回复模式。
language_inputs = batch[:, :-1]  # 用每条序列除最后 token 外的前缀作为训练输入。
language_targets = batch[:, 1:]  # 用右移一位的 token 作为下一词目标。
loss_trace = []  # 保存下一词训练损失轨迹。
first_gradient_norm = 0.0  # 预留首轮 Query 投影梯度范数。
for epoch in range(401):  # 在封闭小语料上执行四百零一次参数更新。
    optimizer.zero_grad()  # 清空上一轮累计梯度。
    logits, cache, attention = gpt(language_inputs)  # 以全量模式运行手写 Decoder。
    supervised_logits = logits[:, 2:, :].reshape(-1, len(vocabulary))  # 只监督意图 token 之后可判定的回复位置。
    supervised_targets = language_targets[:, 2:].reshape(-1)  # 对齐助手、回答动作和对象等目标 token。
    loss = F.cross_entropy(supervised_logits, supervised_targets)  # 计算封闭语料的下一 token 交叉熵。
    loss.backward()  # 反向传播到注意力投影与 embedding。
    if epoch == 0:  # 首轮记录真实注意力梯度规模。
        first_gradient_norm = gpt.block.attention.query_projection.weight.grad.norm().item()  # 读取 Query 投影的首轮梯度范数。
    optimizer.step()  # 根据当前梯度更新模型参数。
    loss_trace.append(loss.item())  # 保存当前轮训练损失。
    if epoch in [0, 50, 150, 400]:  # 选择关键轮次输出学习轨迹。
        token_accuracy = (supervised_logits.argmax(dim=1) == supervised_targets).float().mean().item()  # 计算当前受监督位置的 token 准确率。
        print(f"epoch={epoch:03d} loss={loss.item():.4f} supervised_token_accuracy={token_accuracy:.1%}")  # 输出真实损失与 token 准确率。
gpt.eval()  # 切换到评估模式比较两种解码路径。
print(f"首轮 Query 投影梯度范数={first_gradient_norm:.6f}")  # 输出非零梯度证明模型被真实训练。

epoch=000 loss=2.9447 supervised_token_accuracy=25.0%


epoch=050 loss=0.0001 supervised_token_accuracy=100.0%


epoch=150 loss=0.0000 supervised_token_accuracy=100.0%


epoch=400 loss=0.0000 supervised_token_accuracy=100.0%
首轮 Query 投影梯度范数=0.061868


In [4]:
def decode_full(model, prompt_ids, generation_steps):  # 定义每步重新计算完整前缀的正确性基线。
    sequence = prompt_ids.clone()  # 复制提示词以免修改原始输入。
    generated = []  # 保存每一步生成的 token 编号。
    step_logits = []  # 保存每一步最后位置的词表 logits。
    processed_tokens = 0  # 统计全量重算实际送入模型的 token 数。
    for step in range(generation_steps):  # 按自回归顺序生成指定数量 token。
        logits, cache, attention = model(sequence.unsqueeze(0))  # 把不断增长的完整前缀送入模型。
        current_logits = logits[0, -1].detach().clone()  # 保存当前最后位置的候选 logits。
        next_id = int(torch.argmax(current_logits).item())  # 用贪心策略选择下一个 token。
        step_logits.append(current_logits)  # 保存当前步 logits 供缓存一致性检查。
        generated.append(next_id)  # 保存当前生成 token 编号。
        processed_tokens += sequence.numel()  # 累加本轮完整前缀的计算量。
        sequence = torch.cat([sequence, torch.tensor([next_id], dtype=torch.long)])  # 把新 token 追加到下一轮前缀。
    return generated, step_logits, processed_tokens  # 返回生成结果、逐步 logits 与计算 token 数。
def decode_cached(model, prompt_ids, generation_steps):  # 定义 prefill 后只输入新 token 的 KV cache 解码。
    logits, cache, attention = model(prompt_ids.unsqueeze(0))  # 一次 prefill 处理完整提示词并建立缓存。
    generated = []  # 保存缓存路径生成的 token 编号。
    step_logits = []  # 保存缓存路径每一步的词表 logits。
    processed_tokens = prompt_ids.numel()  # 记录 prefill 处理的提示词 token 数。
    for step in range(generation_steps):  # 按自回归顺序生成指定数量 token。
        current_logits = logits[0, -1].detach().clone()  # 读取当前最后位置的候选 logits。
        next_id = int(torch.argmax(current_logits).item())  # 用与基线相同的贪心策略选择 token。
        step_logits.append(current_logits)  # 保存当前步 logits 供逐元素比较。
        generated.append(next_id)  # 保存当前生成 token 编号。
        if step < generation_steps - 1:  # 仅在还需要下一 token 时继续增量前向。
            next_input = torch.tensor([[next_id]], dtype=torch.long)  # 把刚生成的一个 token 作为新输入。
            logits, cache, attention = model(next_input, cache=cache)  # 复用历史 K/V 并只计算新增 token。
            processed_tokens += 1  # 缓存路径只增加一个被处理 token。
    return generated, step_logits, processed_tokens, cache, attention  # 返回结果、计算量、缓存和末步注意力。
decode_records = []  # 保存六条提示词的两种解码对照。
with torch.no_grad():  # 关闭解码阶段的梯度记录。
    for sequence in token_sequences:  # 逐条客服意图构造五 token 提示词。
        prompt_tokens = sequence[:5]  # 截取到“助手答”作为生成提示词。
        expected_tokens = sequence[5:]  # 保存语料中的两个期望回复 token。
        prompt_ids = torch.tensor([token_to_id[token] for token in prompt_tokens], dtype=torch.long)  # 编码当前可读提示词。
        full_ids, full_logits, full_work = decode_full(gpt, prompt_ids, 2)  # 用全量重算基线生成两个 token。
        cached_ids, cached_logits, cached_work, final_cache, final_attention = decode_cached(gpt, prompt_ids, 2)  # 用 KV cache 路径生成相同长度回复。
        maximum_logit_difference = max((full_logits[step] - cached_logits[step]).abs().max().item() for step in range(2))  # 计算两条路径的最大 logits 差异。
        top_candidates = torch.topk(full_logits[0], k=3)  # 读取第一生成步的前三候选 logits。
        candidate_text = [(id_to_token[index.item()], round(value.item(), 3)) for value, index in zip(top_candidates.values, top_candidates.indices)]  # 把候选编号转换成可读 token 与分数。
        full_tokens = [id_to_token[index] for index in full_ids]  # 解码全量重算路径的 token。
        cached_tokens = [id_to_token[index] for index in cached_ids]  # 解码缓存路径的 token。
        decode_records.append((prompt_tokens, expected_tokens, full_tokens, cached_tokens, full_work, cached_work, maximum_logit_difference, candidate_text))  # 保存完整逐样本对照记录。
print("意图  期望回复      全量生成      Cache生成     token工作量  最大logit差")  # 打印六条解码结果表头。
for record in decode_records:  # 遍历并展示每条客服意图的缓存一致性。
    print(f"{record[0][2]:<4}  {'/'.join(record[1]):<10}  {'/'.join(record[2]):<10}  {'/'.join(record[3]):<10}  {record[4]:>2}->{record[5]:<2}       {record[6]:.2e}")  # 输出期望、两种生成和计算量。
print(f"退款提示第一步 top3={decode_records[0][7]}")  # 展示一个提示词的真实候选 logits 排名。
print(f"末步缓存 K 形状={tuple(final_cache[0].shape)}，注意力形状={tuple(final_attention.shape)}")  # 展示 KV 和注意力中间张量结构。

意图  期望回复      全量生成      Cache生成     token工作量  最大logit差
退款    提交/订单       提交/订单       提交/订单       11->6        5.48e-06
物流    查询/单号       查询/单号       查询/单号       11->6        1.91e-06
密码    重置/账号       重置/账号       重置/账号       11->6        2.86e-06
发票    填写/抬头       填写/抬头       填写/抬头       11->6        1.91e-06
库存    查看/商品       查看/商品       查看/商品       11->6        2.56e-06
优惠    核对/活动       核对/活动       核对/活动       11->6        1.19e-06
退款提示第一步 top3=[('提交', 13.236), ('助手', 1.684), ('单号', 1.501)]
末步缓存 K 形状=(1, 4, 6, 8)，注意力形状=(1, 4, 1, 6)


## 结果解读

全量路径每次处理长度 5、6 的前缀，共 11 个 token；cache 路径只处理 5 个 prefill token 和 1 个新增 token，共 6 个。两者 logits 的微小差异只来自浮点运算顺序，贪心输出必须完全一致。

## 失败案例：增量 token 的位置编号错误地从 0 重启

缓存已有五个位置时，新 token 应使用位置 5。若每步都从位置 0 开始，K/V 虽然拼对了，位置语义却错了，最终 logits 不再等于全量重算。

In [5]:
example_prompt_ids = torch.tensor([token_to_id[token] for token in token_sequences[0][:5]], dtype=torch.long)  # 编码退款案例的五 token 提示词。
with torch.no_grad():  # 关闭梯度以比较正确和错误位置偏移。
    prefill_logits, prefill_cache, prefill_attention = gpt(example_prompt_ids.unsqueeze(0))  # 正确 prefill 并建立长度五的 K/V。
    first_generated_id = int(prefill_logits[0, -1].argmax().item())  # 取得第一个生成 token 作为增量输入。
    incremental_input = torch.tensor([[first_generated_id]], dtype=torch.long)  # 构造只包含新增 token 的输入。
    correct_incremental_logits, correct_cache, correct_attention = gpt(incremental_input, cache=prefill_cache)  # 用缓存长度自动续接位置五。
    wrong_incremental_logits, wrong_cache, wrong_attention = gpt(incremental_input, cache=prefill_cache, position_offset=0)  # 故意把新增 token 位置重置为零。
    full_prefix = torch.cat([example_prompt_ids, torch.tensor([first_generated_id], dtype=torch.long)]).unsqueeze(0)  # 构造包含新增 token 的完整前缀。
    reference_logits, reference_cache, reference_attention = gpt(full_prefix)  # 用全量重算获得正确参考 logits。
correct_position_difference = (correct_incremental_logits[0, -1] - reference_logits[0, -1]).abs().max().item()  # 计算正确 cache 与全量路径差异。
wrong_position_difference = (wrong_incremental_logits[0, -1] - reference_logits[0, -1]).abs().max().item()  # 计算错误位置编号造成的差异。
print(f"正确续接位置的最大 logits 差异={correct_position_difference:.2e}")  # 展示正确缓存路径与全量参考一致。
print(f"错误重置位置的最大 logits 差异={wrong_position_difference:.4f}")  # 展示位置 bug 破坏数值一致性。
print("修复结论：增量位置从 past_length 开始，且要用全量重算逐步做黄金对照。")  # 总结缓存实现的关键验收方法。

正确续接位置的最大 logits 差异=5.48e-06
错误重置位置的最大 logits 差异=12.1480
修复结论：增量位置从 past_length 开始，且要用全量重算逐步做黄金对照。


## 生产差距

真实 GPT 有数十层，每层各自维护 K/V，还要处理批次内不同长度、分页 KV、beam search、请求取消、量化和显存碎片。cache 降低计算却增加随上下文线性增长的显存；线上必须监控 token 延迟、命中率、块利用率和全量/cache 一致性。这里未实现 RoPE、采样策略和多层缓存。

## 最小回归测试

In [6]:
assert len(token_sequences) >= 6  # 保证案例覆盖至少六种真实客服意图。
assert loss_trace[-1] < loss_trace[0]  # 保证真实反向传播使语言模型损失下降。
assert first_gradient_norm > 0.0  # 保证 Query 投影获得了非零梯度。
assert all(record[2] == record[3] for record in decode_records)  # 保证六条提示的全量与 cache 生成 token 完全一致。
assert max(record[6] for record in decode_records) < 1e-4  # 保证两种路径的逐步 logits 数值一致。
assert all(record[5] < record[4] for record in decode_records)  # 保证 cache 路径减少了重复 token 计算。
assert correct_position_difference < 1e-4 and wrong_position_difference > 1e-3  # 保证位置偏移失败可复现且修复有效。
print("回归测试通过：生成、logits、计算量和位置续接均符合预期。")  # 输出集中断言的最终验收结果。

回归测试通过：生成、logits、计算量和位置续接均符合预期。
